# UFC Refresh Notebook

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

from UFCScrapers import Fighter
import UFCScrapers.History as HistoryScraper
import UFCScrapers.Upcoming as UpcomingScraper

In [ ]:
historyPath = "./data/processed/HistoryPost2019.csv"
history2019Path = "./data/processed/HistoryPost2019.csv"
fighterStatsPath = "./data/processed/FighterStats.csv"
skippedFightsPath = "./data/processed/SkippedFights.csv"
upcomingPath = "./data/processed/Upcoming.csv"

if not os.path.exists(historyPath):
    raise FileNotFoundError(f"Missing expected history file: {historyPath}")

dateFormat = "%B %d, %Y"
maxWorkers = 4
startIndex = 0

In [3]:
resultColumns = [
    "r_name", "b_name",
    "r_wins", "r_losses", "r_draws", "r_height", "r_weight", "r_reach",
    "r_splm", "r_str_acc", "r_sapm", "r_str_def",
    "r_td_avg", "r_td_avg_acc", "r_td_def", "r_sub_avg",
    "b_wins", "b_losses", "b_draws", "b_height", "b_weight", "b_reach",
    "b_splm", "b_str_acc", "b_sapm", "b_str_def",
    "b_td_avg", "b_td_avg_acc", "b_td_def", "b_sub_avg",
    "winner_id", "date",
]

skippedColumns = ["r_name", "b_name", "date", "error"]

## 1. Refresh history

In [6]:
# only post 2019 fights for this particular file (wayback does not archive ufc.com before 2019)
historyDf = pd.read_csv(historyPath)
newHistoryDf = HistoryScraper.GetHistory(historyPath, page=1)
historyDf = pd.concat([historyDf, newHistoryDf], ignore_index=True, sort=False)
historyDf = historyDf.drop_duplicates(subset=["r_name", "b_name", "winner_id", "date"], keep="last")
historyDf.to_csv(historyPath, index=False)
historyDf.tail()

,r_name,b_name,winner_id,date
3689,Abdul Rakhman Yakhyaev,Rafael Cerqueira,1.0,"November 22, 2025"
3690,Aleksandre Topuria,Bekzat Almakhan,1.0,"November 22, 2025"
3691,Ismail Naurdiev,Ryan Loder,1.0,"November 22, 2025"
3692,Nurullo Aliev,Shem Rock,1.0,"November 22, 2025"
3693,Denzel Freeman,Marek Bujlo,1.0,"November 22, 2025"


## 3. Scrape fights

In [7]:
def processFight(row):
    rSlug = row.r_name.lower().replace(" ", "-")
    bSlug = row.b_name.lower().replace(" ", "-")
    date = row.date

    try:
        rFighter = Fighter(rSlug, date)
        bFighter = Fighter(bSlug, date)
    except Exception as e:
        return None, {
            "r_name": row.r_name,
            "b_name": row.b_name,
            "date": date,
            "error": str(e),
        }

    result = {
        "r_name": row.r_name,
        "b_name": row.b_name,
        "r_wins": rFighter.wins,
        "r_losses": rFighter.losses,
        "r_draws": rFighter.draws,
        "r_height": rFighter.height,
        "r_weight": rFighter.weight,
        "r_reach": rFighter.reach,
        "r_splm": rFighter.splm,
        "r_str_acc": rFighter.str_acc,
        "r_sapm": rFighter.sapm,
        "r_str_def": rFighter.str_def,
        "r_td_avg": rFighter.td_avg,
        "r_td_avg_acc": rFighter.td_avg_acc,
        "r_td_def": rFighter.td_def,
        "r_sub_avg": rFighter.sub_avg,
        "b_wins": bFighter.wins,
        "b_losses": bFighter.losses,
        "b_draws": bFighter.draws,
        "b_height": bFighter.height,
        "b_weight": bFighter.weight,
        "b_reach": bFighter.reach,
        "b_splm": bFighter.splm,
        "b_str_acc": bFighter.str_acc,
        "b_sapm": bFighter.sapm,
        "b_str_def": bFighter.str_def,
        "b_td_avg": bFighter.td_avg,
        "b_td_avg_acc": bFighter.td_avg_acc,
        "b_td_def": bFighter.td_def,
        "b_sub_avg": bFighter.sub_avg,
        "winner_id": row.winner_id,
        "date": date,
    }
    return result, None

In [ ]:
total = len(historyDf)
completed = 0
skips = 0

print(f"Starting from filtered history index {startIndex}. Processing {total} fights.")

with ThreadPoolExecutor(max_workers=maxWorkers) as executor:
    futures = [executor.submit(processFight, row) for row in historyDf.itertuples(index=False)]

    for future in as_completed(futures):
        result, skipped = future.result()
        completed += 1

        if result is not None:
            pd.DataFrame([result], columns=resultColumns).to_csv(
                fighterStatsPath,
                mode="a",
                header=False,
                index=False,
            )
        else:
            skips += 1
            pd.DataFrame([skipped], columns=skippedColumns).to_csv(
                skippedFightsPath,
                mode="a",
                header=False,
                index=False,
            )

        print(f"Progress: {completed}/{total}, (skipped: {skips})")

Starting from filtered history index 0. Processing 3694 fights.
Progress: 1/3694
Progress: 2/3694
Progress: 3/3694
Progress: 4/3694
Progress: 5/3694
Progress: 6/3694
Progress: 7/3694
Progress: 8/3694
Progress: 9/3694
Progress: 10/3694
Progress: 11/3694
Progress: 12/3694
Progress: 13/3694
Progress: 14/3694
Progress: 15/3694
Progress: 16/3694
Progress: 17/3694
Progress: 18/3694
Progress: 19/3694
Progress: 20/3694
Progress: 21/3694
Progress: 22/3694
Progress: 23/3694
Progress: 24/3694
Progress: 25/3694


## 4. Refresh upcoming card

In [ ]:
upcomingUrl = UpcomingScraper._GetUpcomingURL()
upcomingSoup = UpcomingScraper._GetSoup(upcomingUrl)
upcomingDf = UpcomingScraper.GetCard(upcomingSoup)
upcomingDf.to_csv(upcomingPath, index=False)
upcomingDf